In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.1/29.1 MB 1.3 MB/s eta 0:00:00a 0:00:010m


In [4]:
# Install Open Babel on macOS using Homebrew
!brew install open-babel

==> Auto-updating Homebrew...
Adjust how often this is run with `$HOMEBREW_AUTO_UPDATE_SECS` or disable with
`$HOMEBREW_NO_AUTO_UPDATE=1`. Hide these hints with `$HOMEBREW_NO_ENV_HINTS=1` (see `man brew`).
==> Auto-updated Homebrew!
Updated 3 taps (brewsci/bio, homebrew/core and homebrew/cask).
==> New Formulae
==> Auto-updated Homebrew!
Updated 3 taps (brewsci/bio, homebrew/core and homebrew/cask).
==> New Formulae
atomic_queue: C++14 lock-free queues
attempt-cli: CLI for retrying fallible commands
bedtk: Simple toolset for BED files
claude-cmd: Claude Code Commands Manager
claude-hooks: Hook system for Claude Code
codebook-lsp: Code-aware spell checker language server
context7-mcp: Up-to-date code documentation for LLMs and AI code editors
dash-mpd-cli: Download media content from a DASH-MPEG or DASH-WebM MPD manifest
docker-compose-langserver: Language service for Docker Compose documents
doge: Command-line DNS client
fastmcp: Fast, Pythonic way to build MCP servers and clients
fjir

In [5]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import os

In [7]:
# Load VS-screened results file
df = pd.read_csv("virtual_screening_results.csv")
df.head()

,Common_Name,SMILES,MolWt,LogP,NumHDonors,NumHAcceptors,NumRings,NumRotatableBonds,NumHeavyAtoms,NumHeteroatoms,NumAromaticRings,TPSA,FractionCSP3,MorganFingerprint,Predicted_Prob,Predicted_Activity
0,Ceralifimod,CCCc1ccc(COc2ccc3c(c2)CCC(CN2CC(C(=O)O)C2)=C3C...,435.564,4.96280,1.0,4.0,4.0,9.0,32.0,5.0,2.0,59.00,0.444444,[0 0 0 ... 1 0 0],1.0,1
1,Ponesimod,CCC/N=C1\S/C(=C\c2ccc(OC[C@H](O)CO)c(Cl)c2)C(=...,460.983,4.26732,2.0,6.0,3.0,8.0,31.0,8.0,2.0,82.36,0.304348,[0 1 0 ... 0 0 0],1.0,1
2,Etrasimod,O=C(O)C[C@H]1CCc2c1[nH]c1ccc(OCc3ccc(C4CCCC4)c...,457.492,6.92780,2.0,2.0,5.0,6.0,33.0,7.0,3.0,62.32,0.423077,[0 0 0 ... 0 0 0],1.0,1
3,Siponimod,CCc1cc(/C(C)=N/OCc2ccc(C3CCCCC3)c(C(F)(F)F)c2)...,516.604,6.77270,1.0,4.0,4.0,9.0,37.0,8.0,2.0,62.13,0.517241,[0 0 1 ... 1 0 0],1.0,1
4,Ozanimod,CC(C)Oc1ccc(-c2nc(-c3cccc4c3CC[C@@H]4NCCO)no2)...,404.470,3.63168,2.0,7.0,4.0,7.0,30.0,7.0,3.0,104.20,0.347826,[0 1 0 ... 0 0 0],1.0,1


In [8]:
# Select top 500 based on Predicted Probability
top500 = df.sort_values(by='Predicted_Prob', ascending=False).head(500)
top500.head()

,Common_Name,SMILES,MolWt,LogP,NumHDonors,NumHAcceptors,NumRings,NumRotatableBonds,NumHeavyAtoms,NumHeteroatoms,NumAromaticRings,TPSA,FractionCSP3,MorganFingerprint,Predicted_Prob,Predicted_Activity
0,Ceralifimod,CCCc1ccc(COc2ccc3c(c2)CCC(CN2CC(C(=O)O)C2)=C3C...,435.564,4.96280,1.0,4.0,4.0,9.0,32.0,5.0,2.0,59.00,0.444444,[0 0 0 ... 1 0 0],1.0,1
1,Ponesimod,CCC/N=C1\S/C(=C\c2ccc(OC[C@H](O)CO)c(Cl)c2)C(=...,460.983,4.26732,2.0,6.0,3.0,8.0,31.0,8.0,2.0,82.36,0.304348,[0 1 0 ... 0 0 0],1.0,1
2,Etrasimod,O=C(O)C[C@H]1CCc2c1[nH]c1ccc(OCc3ccc(C4CCCC4)c...,457.492,6.92780,2.0,2.0,5.0,6.0,33.0,7.0,3.0,62.32,0.423077,[0 0 0 ... 0 0 0],1.0,1
3,Siponimod,CCc1cc(/C(C)=N/OCc2ccc(C3CCCCC3)c(C(F)(F)F)c2)...,516.604,6.77270,1.0,4.0,4.0,9.0,37.0,8.0,2.0,62.13,0.517241,[0 0 1 ... 1 0 0],1.0,1
4,Ozanimod,CC(C)Oc1ccc(-c2nc(-c3cccc4c3CC[C@@H]4NCCO)no2)...,404.470,3.63168,2.0,7.0,4.0,7.0,30.0,7.0,3.0,104.20,0.347826,[0 1 0 ... 0 0 0],1.0,1


In [10]:
# Save top 500 to CSV for docking
top500.to_csv("top500_for_docking.csv", index=False)

print("Top 500 compounds saved to top500_for_docking.csv")

Top 500 compounds saved to top500_for_docking.csv


In [11]:
# Load CSV
data = pd.read_csv("top500_for_docking.csv")
data.head()

,Common_Name,SMILES,MolWt,LogP,NumHDonors,NumHAcceptors,NumRings,NumRotatableBonds,NumHeavyAtoms,NumHeteroatoms,NumAromaticRings,TPSA,FractionCSP3,MorganFingerprint,Predicted_Prob,Predicted_Activity
0,Ceralifimod,CCCc1ccc(COc2ccc3c(c2)CCC(CN2CC(C(=O)O)C2)=C3C...,435.564,4.96280,1.0,4.0,4.0,9.0,32.0,5.0,2.0,59.00,0.444444,[0 0 0 ... 1 0 0],1.0,1
1,Ponesimod,CCC/N=C1\S/C(=C\c2ccc(OC[C@H](O)CO)c(Cl)c2)C(=...,460.983,4.26732,2.0,6.0,3.0,8.0,31.0,8.0,2.0,82.36,0.304348,[0 1 0 ... 0 0 0],1.0,1
2,Etrasimod,O=C(O)C[C@H]1CCc2c1[nH]c1ccc(OCc3ccc(C4CCCC4)c...,457.492,6.92780,2.0,2.0,5.0,6.0,33.0,7.0,3.0,62.32,0.423077,[0 0 0 ... 0 0 0],1.0,1
3,Siponimod,CCc1cc(/C(C)=N/OCc2ccc(C3CCCCC3)c(C(F)(F)F)c2)...,516.604,6.77270,1.0,4.0,4.0,9.0,37.0,8.0,2.0,62.13,0.517241,[0 0 1 ... 1 0 0],1.0,1
4,Ozanimod,CC(C)Oc1ccc(-c2nc(-c3cccc4c3CC[C@@H]4NCCO)no2)...,404.470,3.63168,2.0,7.0,4.0,7.0,30.0,7.0,3.0,104.20,0.347826,[0 1 0 ... 0 0 0],1.0,1


In [12]:
# Create folder to save SDFs
output_dir = "Ligand_SDFs"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Loop through each compound
for i, row in data.iterrows():
    name = row['Common_Name']
    smiles = row['SMILES']

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Skipping {name}: Invalid SMILES")
        continue

    mol = Chem.AddHs(mol)

    # Try embedding with ETKDGv3
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    embed_status = AllChem.EmbedMolecule(mol, params)

    # If embedding fails, retry with classic method
    if embed_status != 0:
        embed_status = AllChem.EmbedMolecule(mol, maxAttempts=50, randomSeed=42)

    if embed_status != 0:
        print(f"Skipping {name}: Could not embed 3D coordinates")
        continue

    # Optimize geometry safely
    try:
        if AllChem.MMFFHasAllMoleculeParams(mol):
            AllChem.MMFFOptimizeMolecule(mol)
        else:
            AllChem.UFFOptimizeMolecule(mol)
    except Exception as e:
        print(f"Warning: Could not optimize {name} ({e})")

    # Create safe file name
    safe_name = "".join(c if c.isalnum() else "_" for c in name)
    sdf_path = os.path.join(output_dir, f"{i+1}_{safe_name}.sdf")

    # Write SDF
    writer = Chem.SDWriter(sdf_path)
    writer.write(mol)
    writer.close()

    print(f"Saved {sdf_path}")

In [14]:
!ls Ligand_SDFs | wc -l

     467


In [ ]:
# Convert SDF → PDBQT

# Use OpenBabel or AutoDock Tools (ADT) to convert SDF to PDBQT for Vina docking.

# Example using OpenBabel:

import os

# Paths
input_folder = "Ligand_SDFs"
output_folder = "Ligand_PDBQT"
os.makedirs(output_folder, exist_ok=True)

# Loop through all SDF files
for f in os.listdir(input_folder):
    if f.endswith(".sdf"):
        input_file = os.path.join(input_folder, f)
        base = os.path.splitext(f)[0]
        output_file = os.path.join(output_folder, base + ".pdbqt")
        # Command using OpenBabel
        os.system(f"obabel {input_file} -O {output_file} --gen3d")




In [1]:
import os

folder = "Ligand_PDBQT"   # or "Ligand_SDFs"
files = [f for f in os.listdir(folder) if f.endswith(".pdbqt")]  # change extension if needed
print(f"Number of PDBQT files: {len(files)}")


Number of PDBQT files: 456


In [ ]:
print("First 10 files:", files[:10])


First 10 files: ['458_Aclarubicin.pdbqt', '144_Chrysin.pdbqt', '214_MK_4965.pdbqt', '348_Zibotentan.pdbqt', '468_N__P_CYANOPHENYL__N__DIPHENYLMETHYL_GUANIDINE_ACETIC_ACID.pdbqt', '187_GSK_1292263.pdbqt', '421_Genistein.pdbqt', '449_Lomibuvir.pdbqt', '398_Ro_12_7310.pdbqt', '401_Omeprazole.pdbqt']


In [2]:
import os

input_folder = "Ligand_SDFs"
output_folder = "Ligand_PDBQT"
os.makedirs(output_folder, exist_ok=True)

# Get list of already converted ligands
converted = {os.path.splitext(f)[0] for f in os.listdir(output_folder) if f.endswith(".pdbqt")}

sdf_files = [f for f in os.listdir(input_folder) if f.endswith(".sdf")]
print(f"Total SDF files: {len(sdf_files)}")
print(f"Already converted: {len(converted)}")

# Resume conversion
remaining = 0
for f in sdf_files:
    base = os.path.splitext(f)[0]
    if base in converted:
        continue  # skip already converted
    input_file = os.path.join(input_folder, f)
    output_file = os.path.join(output_folder, base + ".pdbqt")
    os.system(f"obabel {input_file} -O {output_file} --gen3d")
    remaining += 1
    print(f"Converted: {base}")

print(f"✅ Resumed and converted {remaining} new ligands.")


Total SDF files: 467
Already converted: 456


1 molecule converted


Converted: 17_BMS_986104


1 molecule converted


Converted: 216_Ligandrol


...

1 molecule converted


Converted: 226_5_FLUOROINDOLE_PROPANOL_PHOSPHATE


1 molecule converted


Converted: 81_5_Phosphoarabinonic_Acid
Converted: 27_2_Deoxy_2_Amino_Glucitol_6_Phosphate
✅ Resumed and converted 11 new ligands.


1 molecule converted


In [3]:
import os

output_folder = "Ligand_PDBQT"
num_files = len([f for f in os.listdir(output_folder) if f.endswith(".pdbqt")])
print(f"Number of PDBQT files: {num_files}")

Number of PDBQT files: 467


In [ ]:
## Download and zip

!zip -r Ligand_SDFs.zip Ligand_SDFs

In [ ]:
## Download and zip
!zip -r Ligand_PDBQT.zip Ligand_PDBQT

In [ ]:
from google.colab import files

files.download("Ligand_SDFs.zip")

In [ ]:
files.download("Ligand_PDBQT.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>